# 1. Disponibilizar el modelo


In [ ]:
import pandas as pd
import numpy as np
import joblib

In [ ]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2025/main/datasets/dataTrain_Spotify.csv')
dataTesting = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2025/main/datasets/dataTest_Spotify.csv', index_col=0)

In [ ]:
#corrleción con popularity
numericalvars = dataTraining.select_dtypes(include=np.number).columns
dataTraining[numericalvars].corrwith(dataTraining['popularity']).sort_values(ascending=False)

popularity          1.000000
loudness            0.051884
danceability        0.034825
time_signature      0.033124
tempo               0.013556
energy              0.001185
Unnamed: 0         -0.000372
key                -0.002451
liveness           -0.007030
duration_ms        -0.008599
mode               -0.015283
acousticness       -0.027883
valence            -0.041287
speechiness        -0.045089
instrumentalness   -0.095845
dtype: float64

In [ ]:
def newvariables(dataTraining):
    df = dataTraining.copy()
    df['duration_segundos'] = (df['duration_ms'] /1000)+1
    df['density'] = df['loudness'] / df['duration_segundos']
    return df

dataTraining = newvariables(dataTraining)

In [ ]:
dataTraining = dataTraining[['track_genre', 'duration_segundos', 'popularity', 'explicit', 'loudness', 'danceability','time_signature', 'tempo','energy',  'density' ]]

In [ ]:
dataTraining.groupby('track_genre')['popularity'].mean()

group_artist = dataTraining.groupby('track_genre')['popularity'].mean().reset_index().sort_values(by='popularity', ascending=False)

group_10 = group_artist['track_genre'][(group_artist['popularity']>90)&(group_artist['popularity']<101)]
group_9 = group_artist['track_genre'][(group_artist['popularity']>80)&(group_artist['popularity']<91)]
group_8 = group_artist['track_genre'][(group_artist['popularity']>70)&(group_artist['popularity']<81)]
group_7 = group_artist['track_genre'][(group_artist['popularity']>60)&(group_artist['popularity']<71)]
group_6 = group_artist['track_genre'][(group_artist['popularity']>50)&(group_artist['popularity']<61)]
group_5 = group_artist['track_genre'][(group_artist['popularity']>40)&(group_artist['popularity']<51)]
group_4 = group_artist['track_genre'][(group_artist['popularity']>30)&(group_artist['popularity']<41)]
group_3 = group_artist['track_genre'][(group_artist['popularity']>20)&(group_artist['popularity']<31)]
group_2 = group_artist['track_genre'][(group_artist['popularity']>10)&(group_artist['popularity']<21)]
group_1 = group_artist['track_genre'][(group_artist['popularity']>-0.1)&(group_artist['popularity']<11)]


def genre_Group(df, track_genre ):
  df['genre_group'] = np.select(
 [
     df[track_genre].isin(group_10),
     df[track_genre].isin(group_9),
     df[track_genre].isin(group_8),
     df[track_genre].isin(group_7),
     df[track_genre].isin(group_6),
     df[track_genre].isin(group_5),
     df[track_genre].isin(group_4),
     df[track_genre].isin(group_3),
     df[track_genre].isin(group_2),
     df[track_genre].isin(group_1)
 ],
 [
     10,
     9,
     8,
     7,
     6,
     5,
     4,
     3,
     2,
     1
 ],
 default= 4
)


genre_Group(dataTraining, 'track_genre')

In [ ]:
def cleandf(df):
  try:
    df.drop(columns=['Unnamed: 0','track_id' , 'track_name', 'track_genre'], inplace=True)
  except KeyError: # Handle specific KeyError if columns are not found
    pass
  # Return the modified DataFrame

  df.drop(columns =['track_genre'],inplace=True )
  return df



dataTraining = cleandf(dataTraining)


In [ ]:
#escalar
from sklearn.preprocessing import StandardScaler
def escalar(df):

# Seleccionar las columnas numéricas a escalar
  numeric_cols = [ 'duration_segundos', 'loudness', 'danceability', 'tempo', 'energy', 'density' ]

# Crear el StandardScaler
  scaler = StandardScaler()
  df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
  return df

dataTraining = escalar(dataTraining)

In [ ]:
from sklearn.model_selection import train_test_split

x = dataTraining.drop(columns=['popularity'])
y = dataTraining['popularity']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=123)

In [ ]:
"""import numpy as np
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, VotingRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error

VR1 = VotingRegressor(estimators=[('xg', XGBRegressor()), ('rf', RandomForestRegressor()), ('ex', ExtraTreesRegressor())],)
VR1  = VR1.fit(x_train, y_train)
y_pred = VR1.predict(x_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
rmse"""


"import numpy as np\nfrom xgboost import XGBRegressor\nfrom sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, VotingRegressor\nfrom sklearn.model_selection import GridSearchCV, train_test_split\nfrom sklearn.metrics import mean_squared_error\n\nVR1 = VotingRegressor(estimators=[('xg', XGBRegressor()), ('rf', RandomForestRegressor()), ('ex', ExtraTreesRegressor())],)\nVR1  = VR1.fit(x_train, y_train)\ny_pred = VR1.predict(x_test)\nmse = mean_squared_error(y_test, y_pred)\nrmse = np.sqrt(mse)\nrmse"

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

xg = xgb.XGBRegressor(n_estimators=125, random_state=123, gamma =0, learning_rate =0.1, max_depth =18, colsample_bytree = 0.8)
xg.fit(x_train, y_train)

y_pred = xg.predict(x_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)

RMSE: 15.949412482750638


In [ ]:
joblib.dump(xg, 'randomforestMl', compress = 3)

['randomforestMl']

In [ ]:
!pip install flask-restx

In [ ]:
from flask import Flask, request, jsonify
from flask_restx import Api, Resource, fields

app = Flask(__name__)

# Definición API Flask
api = Api(
    app,
    version='1.0',
    title='PrediccionMl',
    description='PrediccionMl')

ns = api.namespace('PrediccionMlXG',
     description='PrediccionMlXG')

# Definición argumentos o parámetros de la API
parser = ns.parser()

parser.add_argument(
  'track_genre',
  type=str,
  required=True,
  help='Ingresa el genero de la canción',
  location='args')

parser.add_argument(
  'duration_ms',
  type=float,
  required=True,
  help='Duraciòn en milisegundos entre 0 y 200000',
  location='args')

parser.add_argument(
  'explicit',
  type=int,
  required=True,
  help='Explicit (1 for yes, 0 for no)',
  location='args')

parser.add_argument(
  'loudness',
  type=float,
  required=True,
  help='Ingrese el loudness entre -50 y 5',
  location='args')

parser.add_argument(
  'danceability',
  type=float,
  required=True,
  help='Ingrese el danceability entre 0 y 1',
  location='args')

parser.add_argument(
  'time_signature',
  type=int,
  required=True,
  help='Ingrese el time_signature [0,1,2,3,4,5]',
  location='args')


parser.add_argument(
  'tempo',
  type=float,
  required=True,
  help='Ingrese el tempo entre 0 y 220',
  location='args')

parser.add_argument(
  'energy',
  type=float,
  required=True,
  help='Ingrese el energy entre 0 y 1',
  location='args')



resource_fields = api.model('Resource', {
    'result': fields.String,
})

In [ ]:
# Disponibilizaciòn del modelo api
# Disponibilizaciòn del modelo api

@ns.route('/')
class PrediccionMlXG(Resource):

 @api.doc(parser=parser)
 @api.marshal_with(resource_fields)
 def get(self):
     args = parser.parse_args()

     # Crear un DataFrame con los datos de entrada
     input_data = pd.DataFrame([args])

     # Asegurarse de que las columnas estén en el mismo orden que durante el entrenamiento
     input_data = input_data[['track_genre', 'duration_ms', 'explicit', 'loudness', 'danceability', 'time_signature', 'tempo', 'energy']]

     # Aplicar las mismas transformaciones que en el entrenamiento
     input_data = newvariables(input_data)  # Crear nuevas variables
     genre_Group(input_data, 'track_genre')  # Agrupar géneros
     input_data = cleandf(input_data)      # Limpiar el DataFrame
     input_data = escalar(input_data)      # Escalar las columnas numéricas

     # Eliminar la columna 'track_genre' ya que no es necesaria para la predicción
     #input_data.drop(columns=['track_genre'], inplace=True)

     # Realizar la predicción
     try:
         prediction = xg.predict(input_data[['duration_segundos', 'explicit', 'loudness', 'danceability',
                                             'time_signature', 'tempo', 'energy', 'density', 'genre_group']])[0]
     except ValueError as e:
         return {"result": f"Error en la predicción: {str(e)}"}, 400

     return {
         "result": f'La popularidad de tu canción es: {prediction}'
     }, 200

if __name__ == '__main__':
  app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.40:5000
Press CTRL+C to quit
127.0.0.1 - - [27/Apr/2025 12:11:21] "GET /PrediccionMlXG/?track_genre=latin&duration_ms=12888&explicit=0&loudness=-12&danceability=0.55&time_signature=3&tempo=220&energy=1 HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2025 12:11:22] "GET /PrediccionMlXG/?track_genre=latin&duration_ms=12888&explicit=0&loudness=-12&danceability=0.55&time_signature=3&tempo=220&energy=1 HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2025 12:11:28] "GET /PrediccionMlXG/?track_genre=pop&duration_ms=12888&explicit=0&loudness=-12&danceability=0.55&time_signature=3&tempo=220&energy=1 HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2025 12:11:43] "GET /PrediccionMlXG/?track_genre=progressive-house&duration_ms=12888&explicit=0&loudness=-12&danceability=0.55&time_signature=3&tempo=220&energy=1 HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2025 12:11:55] "GET /PrediccionMlXG/?track_genre=swedish&duration_ms=12888&exp